# U.S. housing affordability: now vs 2–3 generations ago

This notebook compares median U.S. home prices, median family income, and estimated 30-year fixed mortgage payment burden using public FRED CSV time series (no web search).

In [1]:
import pandas as pd
import numpy as np
from functools import reduce

SERIES = {
    'median_sale_price': 'MSPUS',          # Median sales price of houses sold, quarterly
    'mortgage_rate_30y': 'MORTGAGE30US',  # 30-year fixed mortgage average, weekly
    'cpi': 'CPIAUCSL',                    # CPI-U, monthly
    'median_family_income': 'MEFAINUSA646N' # Median family income, annual
}

def fred_csv(series_id):
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}'
    df = pd.read_csv(url)
    df.columns = ['date', series_id]
    df['date'] = pd.to_datetime(df['date'])
    df[series_id] = pd.to_numeric(df[series_id].replace('.', np.nan), errors='coerce')
    return df.dropna()

raw = {name: fred_csv(sid) for name, sid in SERIES.items()}

# Convert each series to annual averages. For home price, annual average of quarterly median sale price.
annual_parts = []
for name, sid in SERIES.items():
    df = raw[name].copy()
    df['year'] = df['date'].dt.year
    ann = df.groupby('year')[sid].mean().rename(name).reset_index()
    annual_parts.append(ann)

annual = reduce(lambda left, right: left.merge(right, on='year', how='outer'), annual_parts).sort_values('year')

# CPI-adjust home price to latest CPI year available in the merged data.
latest_common_year = int(annual.dropna(subset=['median_sale_price','mortgage_rate_30y','cpi','median_family_income'])['year'].max())
latest_cpi = annual.loc[annual['year'] == latest_common_year, 'cpi'].iloc[0]
annual['home_price_real_latest_dollars'] = annual['median_sale_price'] * latest_cpi / annual['cpi']
annual['price_to_income'] = annual['median_sale_price'] / annual['median_family_income']
annual['down_payment_20pct_to_income'] = 0.20 * annual['median_sale_price'] / annual['median_family_income']

# Mortgage P&I burden: 20% down, 30-year fixed, annualized payment / median family income.
def monthly_payment(principal, annual_rate_pct, n_months=360):
    r = annual_rate_pct / 100 / 12
    return principal * r / (1 - (1 + r) ** (-n_months))

annual['loan_80pct'] = 0.80 * annual['median_sale_price']
annual['monthly_pi'] = monthly_payment(annual['loan_80pct'], annual['mortgage_rate_30y'])
annual['annual_pi_to_income'] = annual['monthly_pi'] * 12 / annual['median_family_income']

# Select latest, about 30 years prior, and about 60 years prior where mortgage-rate data exist.
target_years = [latest_common_year, latest_common_year - 30, latest_common_year - 60]
selected_rows = []
complete = annual.dropna(subset=['median_sale_price','mortgage_rate_30y','cpi','median_family_income','annual_pi_to_income']).copy()
for target in target_years:
    idx = (complete['year'] - target).abs().idxmin()
    selected_rows.append(complete.loc[idx])
selected = pd.DataFrame(selected_rows).drop_duplicates('year').sort_values('year')

# Comparison against latest.
latest = selected[selected['year'] == latest_common_year].iloc[0]
summary = selected.copy()
summary['real_home_price_vs_latest'] = summary['home_price_real_latest_dollars'] / latest['home_price_real_latest_dollars']
summary['price_income_vs_latest'] = summary['price_to_income'] / latest['price_to_income']
summary['pi_burden_vs_latest'] = summary['annual_pi_to_income'] / latest['annual_pi_to_income']

out = summary[[
    'year','median_sale_price','home_price_real_latest_dollars','median_family_income','mortgage_rate_30y',
    'price_to_income','down_payment_20pct_to_income','monthly_pi','annual_pi_to_income',
    'real_home_price_vs_latest','price_income_vs_latest','pi_burden_vs_latest'
]].copy()

formatters = {
    'median_sale_price': '${:,.0f}'.format,
    'home_price_real_latest_dollars': '${:,.0f}'.format,
    'median_family_income': '${:,.0f}'.format,
    'mortgage_rate_30y': '{:.2f}%'.format,
    'price_to_income': '{:.2f}x'.format,
    'down_payment_20pct_to_income': '{:.0%}'.format,
    'monthly_pi': '${:,.0f}'.format,
    'annual_pi_to_income': '{:.0%}'.format,
    'real_home_price_vs_latest': '{:.2f}x'.format,
    'price_income_vs_latest': '{:.2f}x'.format,
    'pi_burden_vs_latest': '{:.2f}x'.format,
}
print('Latest common year:', latest_common_year)
display(out.style.format(formatters))

# Print direct latest / past ratios for easy reading.
for _, row in selected[selected['year'] != latest_common_year].iterrows():
    print(f"\nLatest ({latest_common_year}) vs {int(row['year'])}:")
    print(f"  Real median sale price: {latest['home_price_real_latest_dollars']/row['home_price_real_latest_dollars']:.2f}x as high")
    print(f"  Price / median family income: {latest['price_to_income']/row['price_to_income']:.2f}x as high")
    print(f"  20% down payment / income: {latest['down_payment_20pct_to_income']/row['down_payment_20pct_to_income']:.2f}x as high")
    print(f"  Mortgage P&I / income: {latest['annual_pi_to_income']/row['annual_pi_to_income']:.2f}x as high")


Latest common year: 2024


,year,median_sale_price,home_price_real_latest_dollars,median_family_income,mortgage_rate_30y,price_to_income,down_payment_20pct_to_income,monthly_pi,annual_pi_to_income,real_home_price_vs_latest,price_income_vs_latest,pi_burden_vs_latest
24,1971.000000,"$25,225","$195,464","$10,290",7.54%,2.45x,49%,$142,17%,0.47x,0.62x,0.67x
47,1994.000000,"$130,425","$276,027","$38,780",8.38%,3.36x,67%,$793,25%,0.66x,0.85x,1.00x
77,2024.000000,"$418,975","$418,975","$105,800",6.72%,3.96x,79%,"$2,168",25%,1.00x,1.00x,1.00x



Latest (2024) vs 1971:
  Real median sale price: 2.14x as high
  Price / median family income: 1.62x as high
  20% down payment / income: 1.62x as high
  Mortgage P&I / income: 1.49x as high

Latest (2024) vs 1994:
  Real median sale price: 1.52x as high
  Price / median family income: 1.18x as high
  20% down payment / income: 1.18x as high
  Mortgage P&I / income: 1.00x as high
